##Event log manipulation

This code creates the actor behavior enriched datasets. Start from the original raw BPIC event logs. The code then adds a column that decides upon the actor behavior between two consecutive events in a process intance (case).


In [ ]:
!pip install polars tqdm
from tqdm import tqdm


In [ ]:
def classify_actor_behavior(df_dict):
    """ Determines actor behavior classification"""
    classified_edges = []
    actor_events = {}
    df = pl.DataFrame(df_dict)
    df = df.sort(["case_id", "timestamp"])

     #Take two consecutive events
    for i in tqdm(range(len(df) - 1), desc="Processing events", unit="pairs"):
        e_i, e_j = df.row(i, named=True), df.row(i + 1, named=True)

        # Check if the two consecutive events belong to the same case
        if e_i["case_id"] != e_j["case_id"]:
            continue

       # Retrieve actors and timestamps
        r_i, r_j = e_i["actor"], e_j["actor"]
        t_i, t_j = e_i["timestamp"], e_j["timestamp"]
        delta_t = (t_j - t_i).total_seconds() #time between two consecutive events

        # Same actor → Continuation or Interruption
        if r_i == r_j:
            concurrent_cases = any(t_i < e["timestamp"] < t_j for e in actor_events.get(r_i, []))
            # event from the dataframa where the actor equals r_i (=r_j is) and the timestamp lies in between ti and tj
#decision:
#If concurrent_cases NOT empty → Interruption (I).
#If concurrent_cases empty → Continuation (C).
            behavior = "I" if concurrent_cases else "C"
        else:
            # Different actor → Handover
            concurrent_cases = any(t_i < e["timestamp"] < t_j for e in actor_events.get(r_j, [])) # event from the dataframe where the actor is r_j and the timestamp lies between ti and tj

            if not concurrent_cases:
                behavior = "HI"
            else:
                behavior =  "HB"


        # add result
        classified_edges.append((e_i["case_id"], e_i["event"], t_i, e_j["event"], t_j, r_i, r_j, behavior, delta_t))#, e_i["Accepted"], e_j["Accepted"]))

        actor_events.setdefault(r_i, []).append({"timestamp": t_i, "case_id": e_i["case_id"]})
        actor_events.setdefault(r_j, []).append({"timestamp": t_j, "case_id": e_j["case_id"]})


    return classified_edges


In [ ]:
import pandas as pd
from tqdm import tqdm

input_file = "BPIC12.csv"
output_file = "BPIC2012_actor_behavior_results.csv"

df = pd.read_csv(input_file, dtype=str)
df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True, errors="coerce")
df = df.dropna(subset=["case_id", "event", "actor", "timestamp"])
df = df.sort_values(["case_id", "timestamp"]).reset_index(drop=True)

results = classify_actor_behavior(df)

ab = pd.DataFrame(results, columns=[
    "case_id", "event_i", "timestamp_i", "event_j", "timestamp_j",
    "actor_i", "actor_j", "behavior", "delta_t"
])

ab.to_csv(output_file, index=False)
print(ab.head())
print(ab["behavior"].value_counts())

In [ ]:
from google.colab import files
files.download("BPIC2012_actor_behavior_results.csv")


visualizations

In [ ]:
!pip install matplotlib seaborn networkx


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import networkx as nx

df = pd.read_csv("BPIC2011_actor_behavior_results.csv", delimiter=",", on_bad_lines="skip")

# visualize behacior
plt.figure(figsize=(8, 5))
sns.countplot(data=df, x="behavior", order=["C", "I", "HI", "HB"], palette="coolwarm")
plt.xlabel("Behavior type")
plt.ylabel("Number")
plt.title("Distribution of of behavior types")
plt.show()

# delta_t visualization
plt.figure(figsize=(8, 5))
sns.histplot(df["delta_t"], bins=50, kde=True, color="purple")
plt.xlabel("Time between events")
plt.ylabel("Number")
plt.title("Distribution of Time between Events")
plt.xlim(0, df["delta_t"].quantile(0.99))
plt.show()



In [ ]:
import pandas as pd

print(df.head(10))


In [ ]:
from IPython.display import display

display(df.head(10))
